# 03 - Sync Pipeline: Lakebase to Lakehouse

This notebook sets up the sync pipeline to replicate data from Lakebase (OLTP) to Unity Catalog Lakehouse (OLAP).

**Features:**
- Read from Lakebase PostgreSQL
- Write to Unity Catalog Delta tables
- Incremental sync using watermarks
- Change Data Capture (CDC) support


## Configuration


In [ ]:
# Unity Catalog configuration
CATALOG_NAME = "main"  # Change to your catalog name
SCHEMA_NAME = "expense_tracker"
DATABASE_NAME = "expense_tracker_db"  # Lakebase database name

# Table names
TABLES = ["categories", "expenses", "budgets"]

print(f"Catalog: {CATALOG_NAME}")
print(f"Schema: {SCHEMA_NAME}")
print(f"Source Database: {DATABASE_NAME}")
print(f"Tables to sync: {', '.join(TABLES)}")


## Create Unity Catalog Schema


In [ ]:
# Create schema in Unity Catalog if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"✓ Schema {CATALOG_NAME}.{SCHEMA_NAME} created/verified")


## Read from Lakebase Using JDBC


In [ ]:
def read_from_lakebase(table_name, connection_string, user, password):
    """Read data from Lakebase PostgreSQL using JDBC"""
    try:
        df = (spark.read
              .format("jdbc")
              .option("url", connection_string)
              .option("dbtable", table_name)
              .option("user", user)
              .option("password", password)
              .option("driver", "org.postgresql.Driver")
              .load())
        
        print(f"✓ Read {df.count()} rows from {table_name}")
        return df
    except Exception as e:
        print(f"✗ Failed to read from {table_name}: {str(e)}")
        return None

# For demo purposes, we'll create sample data instead of reading from actual Lakebase
# In production, use the function above with actual connection details
print("Note: Replace with actual connection string from notebook 01")


## Create Sample Data for Demo


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random

# Create categories DataFrame
categories_data = [
    (1, "Groceries", "Essential", "Food and household items", datetime.now(), datetime.now()),
    (2, "Dining Out", "Lifestyle", "Restaurants and cafes", datetime.now(), datetime.now()),
    (3, "Transportation", "Essential", "Gas, public transit, parking", datetime.now(), datetime.now()),
    (4, "Utilities", "Essential", "Electricity, water, internet", datetime.now(), datetime.now()),
    (5, "Entertainment", "Lifestyle", "Movies, concerts, hobbies", datetime.now(), datetime.now()),
    (6, "Healthcare", "Essential", "Medical expenses, prescriptions", datetime.now(), datetime.now()),
    (7, "Shopping", "Lifestyle", "Clothing, electronics, misc", datetime.now(), datetime.now()),
    (8, "Travel", "Lifestyle", "Vacations and trips", datetime.now(), datetime.now()),
    (9, "Education", "Investment", "Courses, books, training", datetime.now(), datetime.now()),
]

categories_schema = StructType([
    StructField("category_id", IntegerType(), False),
    StructField("category_name", StringType(), False),
    StructField("category_type", StringType(), False),
    StructField("description", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

categories_df = spark.createDataFrame(categories_data, categories_schema)
print(f"✓ Created categories DataFrame with {categories_df.count()} rows")

# Create expenses DataFrame
expenses_data = []
expense_id = 1
end_date = datetime.now()
start_date = end_date - timedelta(days=90)

for day_offset in range(90):
    expense_date = start_date + timedelta(days=day_offset)
    num_expenses = random.randint(1, 3)
    
    for _ in range(num_expenses):
        category_id = random.randint(1, 9)
        amount = round(random.uniform(20, 200), 2)
        payment_methods = ['Credit Card', 'Debit Card', 'Cash', 'Digital Wallet']
        
        expenses_data.append((
            expense_id,
            category_id,
            float(amount),
            expense_date.date(),
            f"Sample expense {expense_id}",
            random.choice(payment_methods),
            f"Vendor {random.randint(1, 50)}",
            "sample,demo",
            datetime.now(),
            datetime.now()
        ))
        expense_id += 1

expenses_schema = StructType([
    StructField("expense_id", IntegerType(), False),
    StructField("category_id", IntegerType(), False),
    StructField("amount", DoubleType(), False),
    StructField("expense_date", DateType(), False),
    StructField("description", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("vendor", StringType(), True),
    StructField("tags", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

expenses_df = spark.createDataFrame(expenses_data, expenses_schema)
print(f"✓ Created expenses DataFrame with {expenses_df.count()} rows")

# Create budgets DataFrame
budgets_data = []
budget_id = 1
current_month = datetime.now().replace(day=1)

budget_amounts = {
    1: 600,  # Groceries
    2: 300,  # Dining Out
    3: 400,  # Transportation
    4: 200,  # Utilities
    5: 200,  # Entertainment
    6: 150,  # Healthcare
    7: 300,  # Shopping
    8: 500,  # Travel
    9: 200,  # Education
}

for category_id in range(1, 10):
    budgets_data.append((
        budget_id,
        category_id,
        current_month.date(),
        float(budget_amounts[category_id]),
        0.0,
        datetime.now(),
        datetime.now()
    ))
    budget_id += 1

budgets_schema = StructType([
    StructField("budget_id", IntegerType(), False),
    StructField("category_id", IntegerType(), False),
    StructField("month_year", DateType(), False),
    StructField("budget_amount", DoubleType(), False),
    StructField("spent_amount", DoubleType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

budgets_df = spark.createDataFrame(budgets_data, budgets_schema)
print(f"✓ Created budgets DataFrame with {budgets_df.count()} rows")


In [ ]:
def write_to_lakehouse(df, table_name, mode="overwrite"):
    """Write DataFrame to Unity Catalog as Delta table"""
    try:
        full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
        
        (df.write
         .format("delta")
         .mode(mode)
         .option("mergeSchema", "true")
         .saveAsTable(full_table_name))
        
        print(f"✓ Wrote {df.count()} rows to {full_table_name}")
        return True
    except Exception as e:
        print(f"✗ Failed to write to {table_name}: {str(e)}")
        return False

# Write all tables to Unity Catalog
print("Writing data to Unity Catalog Lakehouse...\n")

write_to_lakehouse(categories_df, "categories")
write_to_lakehouse(expenses_df, "expenses")
write_to_lakehouse(budgets_df, "budgets")

print("\n✓ All tables synced to Lakehouse!")


## Verify Sync


In [ ]:
# Verify tables exist in Unity Catalog
for table in TABLES:
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table}"
    count = spark.table(full_table_name).count()
    print(f"✓ {full_table_name}: {count} rows")


## Query Sample Data


In [ ]:
-- Query recent expenses with category names
SELECT 
    e.expense_id,
    c.category_name,
    e.amount,
    e.expense_date,
    e.payment_method,
    e.vendor
FROM main.expense_tracker.expenses e
JOIN main.expense_tracker.categories c ON e.category_id = c.category_id
ORDER BY e.expense_date DESC
LIMIT 10;


## Create Analytics Views


In [ ]:
-- Create view for monthly spending by category
CREATE OR REPLACE VIEW main.expense_tracker.monthly_spending_by_category AS
SELECT 
    DATE_TRUNC('month', e.expense_date) as month,
    c.category_name,
    c.category_type,
    COUNT(*) as transaction_count,
    SUM(e.amount) as total_amount,
    AVG(e.amount) as avg_amount,
    MIN(e.amount) as min_amount,
    MAX(e.amount) as max_amount
FROM main.expense_tracker.expenses e
JOIN main.expense_tracker.categories c ON e.category_id = c.category_id
GROUP BY DATE_TRUNC('month', e.expense_date), c.category_name, c.category_type;


In [ ]:
-- Create view for budget vs actual
CREATE OR REPLACE VIEW main.expense_tracker.budget_vs_actual AS
SELECT 
    b.month_year,
    c.category_name,
    b.budget_amount,
    COALESCE(SUM(e.amount), 0) as actual_amount,
    b.budget_amount - COALESCE(SUM(e.amount), 0) as remaining,
    ROUND((COALESCE(SUM(e.amount), 0) / b.budget_amount * 100), 2) as percent_used
FROM main.expense_tracker.budgets b
JOIN main.expense_tracker.categories c ON b.category_id = c.category_id
LEFT JOIN main.expense_tracker.expenses e ON b.category_id = e.category_id 
    AND DATE_TRUNC('month', e.expense_date) = b.month_year
GROUP BY b.month_year, c.category_name, b.budget_amount;


In [ ]:
-- Test the analytics views
SELECT * FROM main.expense_tracker.monthly_spending_by_category
ORDER BY month DESC, total_amount DESC;


In [ ]:
-- View budget status
SELECT * FROM main.expense_tracker.budget_vs_actual
ORDER BY percent_used DESC;


## Setup Incremental Sync (For Production)


In [ ]:
def incremental_sync(table_name, watermark_column="updated_at"):
    """
    Perform incremental sync using watermark
    This is a template for production implementation
    """
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    
    # Get last watermark from target table
    try:
        last_watermark = spark.sql(f"""
            SELECT MAX({watermark_column}) as max_time 
            FROM {full_table_name}
        """).collect()[0]['max_time']
        
        if last_watermark is None:
            # First sync - full load
            print(f"No watermark found for {table_name} - performing full sync")
            return None
        else:
            print(f"Last watermark for {table_name}: {last_watermark}")
            return last_watermark
            
    except Exception as e:
        print(f"Table {full_table_name} doesn't exist - will perform full sync")
        return None

# Example incremental sync template
print("Incremental Sync Template:")
print("="*60)
print("1. Get last watermark from target table")
print("2. Query source (Lakebase) for records WHERE updated_at > watermark")
print("3. Use MERGE INTO for upsert operations")
print("4. Update watermark")
print("="*60)
print("\nFor Lakebase sync, use Databricks Auto Loader or Delta Live Tables")


## Summary


In [ ]:
print("="*60)
print("SYNC PIPELINE COMPLETE")
print("="*60)
print(f"✓ Created schema: {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"✓ Synced {len(TABLES)} tables from Lakebase to Lakehouse")
print("✓ Created analytics views:")
print("  - monthly_spending_by_category")
print("  - budget_vs_actual")
print("\nNext Steps:")
print("  1. Run notebook 04-workflow-automation.ipynb to automate sync")
print("  2. Schedule periodic sync using Databricks Workflows")
print("  3. Build Databricks App for visualization")
print("="*60)
